# Small Box Review Analysis

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from matplotlib.ticker import PercentFormatter
import numpy as np
from scipy.stats import gaussian_kde

In [2]:
import os
import sys

import io
from io import BytesIO
import csv

import google.auth
from google.cloud import bigquery
#from google.cloud import bigquery_storage

In [5]:
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "../market-analysis-project-91130-5213911f50a5.json"

credentials, project_id = google.auth.default(
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)

bqclient = bigquery.Client(credentials=credentials, project=project_id,)

In [7]:
# get the data from BigQuery
sql = f"""
select * from wook.review_sales_by_collection_category_year 
"""

df = bqclient.query(sql).to_dataframe()

In [9]:
print(df)

        yr financial_category           main_collection  written_avg_rating  \
0     2022        Box Springs                 BIFDA 4in            5.000000   
1     2022        Box Springs                 Josie 5in            4.750000   
2     2022        Box Springs        Jayanna BIFD 7.5in            4.725000   
3     2022        Box Springs              Victor 7.5in            4.717391   
4     2022        Box Springs          Jayanna BIFD 4in            4.704545   
...    ...                ...                       ...                 ...   
1970  2025            Toppers          3in Swirl Copper                 NaN   
1971  2025            Toppers          1in GT Zoned Gel                 NaN   
1972  2025            Toppers  1.5in Copper Convoluted                  NaN   
1973  2025            Toppers         2in Cloud w Cover                 NaN   
1974  2025            Toppers         1.5in Rejuvenator                 NaN   

      written_12_cnt  written_all_cnt  written_12_r

In [17]:
df1_total = df[df['main_collection'] == '__TOTAL__'].copy()

In [33]:
print(df1_total)

        yr     financial_category origin_collection main_collection  \
0     2022            Box Springs         __TOTAL__       __TOTAL__   
36    2022        Foam Mattresses         __TOTAL__       __TOTAL__   
105   2022  Non Bedroom Furniture         __TOTAL__       __TOTAL__   
184   2022    Other Frames & Beds         __TOTAL__       __TOTAL__   
222   2022                 Others         __TOTAL__       __TOTAL__   
244   2022          Platform Beds         __TOTAL__       __TOTAL__   
435   2022             SmartBases         __TOTAL__       __TOTAL__   
456   2022                   Sofa         __TOTAL__       __TOTAL__   
480   2022      Spring Mattresses         __TOTAL__       __TOTAL__   
540   2022                Toppers         __TOTAL__       __TOTAL__   
581   2023            Box Springs         __TOTAL__       __TOTAL__   
617   2023        Foam Mattresses         __TOTAL__       __TOTAL__   
695   2023  Non Bedroom Furniture         __TOTAL__       __TOTAL__   
779   

In [47]:
agg_df = (
    df1_total
    .groupby(['financial_category', 'yr'], as_index=False)
    .agg( 
        written_avg_rating=('written_avg_rating', 'mean'),
        all_avg_rating=('all_avg_rating', 'mean')
    )
    .round(2)
)
print(agg_df)

       financial_category    yr  written_avg_rating  all_avg_rating
0             Box Springs  2022                4.13            4.52
1             Box Springs  2023                4.11            4.48
2             Box Springs  2024                4.01            4.39
3             Box Springs  2025                3.67            4.09
4         Foam Mattresses  2022                3.53            4.43
5         Foam Mattresses  2023                3.56            4.33
6         Foam Mattresses  2024                3.55            4.22
7         Foam Mattresses  2025                3.34            3.77
8   Non Bedroom Furniture  2022                4.12            4.51
9   Non Bedroom Furniture  2023                4.10            4.47
10  Non Bedroom Furniture  2024                3.84            4.41
11  Non Bedroom Furniture  2025                3.99            4.34
12    Other Frames & Beds  2022                4.16            4.59
13    Other Frames & Beds  2023                4

In [49]:
pivot_df = (
    agg_df.pivot(
        index = 'financial_category',
        columns='yr',
        values=['written_avg_rating','all_avg_rating']
    )
)

pivot_df

written_avg_rating                   all_avg_rating  \
yr                                  2022  2023  2024  2025           2022   
financial_category                                                          
Box Springs                         4.13  4.11  4.01  3.67           4.52   
Foam Mattresses                     3.53  3.56  3.55  3.34           4.43   
Non Bedroom Furniture               4.12  4.10  3.84  3.99           4.51   
Other Frames & Beds                 4.16  4.01  4.05  3.70           4.59   
Others                              4.41  4.34  4.27  3.94           4.49   
Platform Beds                       4.14  4.04  3.94  3.62           4.61   
SmartBases                          4.11  4.16  3.94  3.52           4.59   
Sofa                                3.89  4.03  3.78  3.42           4.23   
Spring Mattresses                   3.73  3.74  3.45  3.43           4.37   
Toppers                             2.79  2.93  3.32  3.03           4.20   

                                         
yr                     2023  2024  2025  
financial_category                       
Box Springs            4.48  4.39  4.09  
Foam Mattresses        4.33  4.22  3.77  
Non Bedroom Furniture  4.47  4.41  4.34  
Other Frames & Beds    4.49  4.48  4.27  
Others                 4.53  4.55  4.31  
Platform Beds          4.51  4.38  4.06  
SmartBases             4.59  4.53  4.03  
Sofa                   4.27  4.26  4.13  
Spring Mattresses      4.33  4.10  3.83  
Toppers                4.07  4.12  3.95

In [69]:
# category별 data -> csv 출력
df1_total = df1_total.sort_values(
    by=['financial_category', 'yr'], ascending=[True, True]
)

df1_total.to_csv('review_sales_total_0108.csv', index=False)

In [39]:
df1 = df[df['main_collection'] != '__TOTAL__'].copy()

In [43]:
df1['is_wonder'] = (
    df1['main_collection'].str.contains('(+', regex=False, na=False)
    .astype(int)
)

In [45]:
print(df1)

        yr financial_category         origin_collection  \
1     2022        Box Springs                Walter 9in   
2     2022        Box Springs              Walter 7.5in   
3     2022        Box Springs                Walter 4in   
4     2022        Box Springs                Victor 9in   
5     2022        Box Springs              Victor 7.5in   
...    ...                ...                       ...   
2562  2025            Toppers               1.5in GT MF   
2563  2025            Toppers    1.5in GTFT w WonderBox   
2564  2025            Toppers  1.5in Copper Convoluted    
2565  2025            Toppers       1.25in Swirl Copper   
2566  2025            Toppers        0.75in Rejuvenator   

               main_collection  written_avg_rating  written_12_cnt  \
1                   Walter 9in            4.239130              16   
2                 Walter 7.5in            4.207407              21   
3                   Walter 4in            4.473333              15   
4          

In [49]:
df1['is_wonder'].sum()

363

In [ ]:
df

In [51]:
df1.to_csv('review_sales_wbox_0108.csv', index=False)

In [55]:
df1_sorted = df1.sort_values(
    by=['financial_category', 'main_collection', 'origin_collection', 'yr'],
    ascending=[True, True, True, True]
)
df1_sorted.to_csv('review_sales_wbox_0108-1.csv', index=False)